# One-country pipeline walkthrough

Runs `backend/utils/pipeline._process_country` one step at a time, so every
intermediate is visible: the macro panel, the LLM payload, the raw article pool,
the stage-1 digests, the model's subscores and per-article impacts, and the
Top-3 that reach the dashboard.

**Pick a country by editing `ISO2` in the cell under _Pick a country_, then Run All.**

Before the first run, install a kernel into the project venv (it is deliberately
not in `requirements.txt`, which is runtime-only):

```
.venv\Scripts\python.exe -m pip install ipykernel
```

Two things to know:

- **This writes no snapshot.** The last step builds the payload
  `data_push.upsert_snapshot` would take and prints it, but does not call it.
  The one thing that does touch Postgres is the stage-1 digest cache in Step 2b
  (`article_digest`), which is scratch data keyed by article URL — nothing the
  dashboard reads.
- **It makes real network calls**, and **Steps 2b and 3 spend OpenAI credits**
  (one cheap digest call per article, then one scoring call). A country with no
  local macro panel also hits the World Bank once per indicator in Step 0, which
  is slow.

## Setup

Resolves the repo root, loads `backend/.env`, and routes the pipeline's logging
into the notebook. `force=True` on `basicConfig` is not optional — Jupyter
installs its own root handler, so a plain `basicConfig()` is silently a no-op
and no pipeline logs appear.

In [ ]:
import json
import logging
import os
import pathlib
import sys

import pandas as pd
from dotenv import load_dotenv

# Repo root = the folder holding backend/main.py, so this works whether the
# kernel's cwd is backend/ or the repo root.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Same two-line load as main.py; every module reads os.getenv at call time.
load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

print(f"project root: {PROJECT_ROOT}")
for key in ("OPENAI_API_KEY", "CRAWLBASE_TOKEN", "CRAWLBASE_JS_TOKEN"):
    print(f"  {key:<20} {'set' if os.getenv(key) else 'MISSING'}")
print(f"  {'DATABASE_URL':<20} "
      f"{'set - used by the Step 2b digest cache only' if os.getenv('DATABASE_URL') else 'MISSING - Step 2b just skips its cache'}")

project root: D:\Coding\ai-country-risk
  OPENAI_API_KEY       set
  CRAWLBASE_TOKEN      set
  CRAWLBASE_JS_TOKEN   set
  DATABASE_URL         set - used by the Step 2b digest cache only


In [ ]:
from backend.utils import constants, data_retrieval
from backend.utils.ai import client as ai_client
from backend.utils.ai import digest_engine, langchain_llm
from backend.utils.data_fetching import country_data_fetch
from backend.utils.data_upsert import data_push
from backend.utils.news_fetching import article_enrichment, article_ranking

# Payload window and article cap, mirrored from backend/utils/pipeline.py:32-37.
SINCE_YEAR = 2015
LOOKBACK_YEARS = 10
DELTA_HORIZONS = (1, 5)
MAX_ARTICLES = 20

print(f"pipeline modules loaded - scoring model: {ai_client.MODEL_NAME}, "
      f"digest model: {ai_client.DIGEST_MODEL_NAME}")

pipeline modules loaded - scoring model: gpt-4o-2024-08-06, digest model: gpt-4o-mini-2024-07-18


## Pick a country

The table below is `constants.COUNTRY_ROSTER`. `has_panel` says whether the
macro panel is already on disk — those countries skip the slow World Bank
backfill in Step 0.

In [ ]:
roster = pd.DataFrame(constants.COUNTRY_ROSTER)
roster["has_panel"] = roster["iso2"].map(
    lambda c: country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, c)
)

print(f"{len(roster)} countries, {int(roster.has_panel.sum())} with a local macro panel")
roster.sort_values(["has_panel", "tier", "name"], ascending=[False, True, True])

48 countries, 5 with a local macro panel


,name,iso2,iso3,tier,lat,lng,has_panel
14,New Zealand,NZ,NZL,DM,-41.5000,173.0000,True
16,Portugal,PT,PRT,DM,39.7000,-8.0000,True
22,United States,US,USA,DM,39.7500,-100.5000,True
27,Czechia,CZ,CZE,EM,49.8200,15.4700,True
43,Taiwan,TW,TWN,EM,23.7000,120.9600,True
0,Australia,AU,AUS,DM,-24.6809,134.5300,False
1,Austria,AT,AUT,DM,47.6082,14.3738,False
2,Belgium,BE,BEL,DM,50.6003,4.7000,False
3,Canada,CA,CAN,DM,60.9215,-108.0070,False
4,Denmark,DK,DNK,DM,55.6761,10.5683,False


In [ ]:
ISO2 = "PT"   # <-- change country here

ENTRY = next(c for c in constants.COUNTRY_ROSTER if c["iso2"] == ISO2)
NAME, ISO3 = ENTRY["name"], ENTRY["iso3"]

print(f"{NAME}  ({ISO2}/{ISO3})  tier={ENTRY['tier']}  map=({ENTRY['lat']}, {ENTRY['lng']})")

Portugal  (PT/PRT)  tier=DM  map=(39.7, -8.0)


## Step 0 — macro panel

Each country's World Bank indicators live in a Parquet partition under
`backend/data/wb_panel_wide/country_code=XX/`. If this country has none, the
real backfill runs with the roster temporarily narrowed to this one entry — the
same trick `backend/tests/live_country_check.py` uses. Expect a few minutes and
one World Bank call per indicator.

The panel is then read back through DuckDB, exactly as the pipeline reads it.

In [ ]:
if country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, ISO2):
    print(f"panel already on disk for {ISO2}")
else:
    print(f"no panel for {ISO2} - building it (slow: one World Bank call per indicator)")
    full_roster = constants.COUNTRY_ROSTER
    constants.COUNTRY_ROSTER = [ENTRY]
    try:
        country_data_fetch.backfill_missing_panels()
    finally:
        constants.COUNTRY_ROSTER = full_roster

panel = data_retrieval.query_macro_panel(ISO2)
print(f"panel: {panel.shape[0]} years x {panel.shape[1]} columns")
panel.tail(15)

panel already on disk for PT
panel: 26 years x 11 columns


,year,INFLATION,UNEMPLOYMENT,FDI_PCT_GDP,POL_STABILITY,RULE_OF_LAW,GINI_INDEX,GDP_PC_GROWTH,INT_PAYM_PCT_REV,POL_CORRUPTION,country_code
11,2011,3.653011,12.715,4.236488,0.783900,0.867073,36.3,-1.568857,11.755316,0.091,PT
12,2012,2.773339,15.509,7.215352,0.841201,0.888092,36.0,-3.661115,12.959271,0.087,PT
13,2013,0.274417,16.237,6.436543,0.763400,0.900620,36.2,-0.439198,12.157183,0.091,PT
14,2014,-0.278153,13.880,5.438510,0.940371,1.018187,35.6,1.286046,12.409146,0.091,PT
15,2015,0.487939,12.405,0.635222,1.000434,1.127288,35.5,2.011396,11.739378,0.093,PT
16,2016,0.607397,11.058,3.562830,0.958808,1.115692,35.2,2.326431,10.823892,0.104,PT
17,2017,1.368614,8.878,5.020416,1.054818,1.172268,33.8,3.567001,9.951020,0.094,PT
18,2018,0.993716,6.966,3.465843,1.091740,1.130340,33.5,3.111237,8.748283,0.100,PT
19,2019,0.338178,6.422,4.503894,0.994125,1.116551,32.8,2.721302,7.744142,0.105,PT
20,2020,-0.012438,6.850,1.814146,0.983046,1.183053,34.7,-8.301071,7.420530,0.121,PT


## Step 1 — the LLM payload

`prepare_llm_payload_pretty` compresses that panel into what the prompt actually
sees: per indicator a latest value, 1-year and 5-year percent changes, and the
last 10 observations. `_meta.generated_at` is the timestamp that would become
the snapshot's `as_of`.

`ALL_INDICATORS` is the World Bank set plus the OWID Political Corruption Index,
merged in at ingest time.

In [ ]:
payload = data_retrieval.prepare_llm_payload_pretty(
    country_iso=ISO2,
    indicators=constants.ALL_INDICATORS,
    since=SINCE_YEAR,
    lookback=LOOKBACK_YEARS,
    deltas=DELTA_HORIZONS,
)

units = payload["_meta"]["units"]
indicators = pd.DataFrame([
    {
        "indicator": name,
        "unit": units.get(name, ""),
        "latest": v["latest"],
        "\u03941y": v["\u03941y"],
        "\u03945y": v["\u03945y"],
        "years": len(v["series"]),
    }
    for name, v in payload["indicators"].items()
]).set_index("indicator")

print(f"latest_year={payload['latest_year']}  generated_at={payload['_meta']['generated_at']}")
indicators

latest_year=2025  generated_at=2026-07-26T22:05Z


,unit,latest,Δ1y,Δ5y,years
indicator,,,,,
Inflation (% y/y),% y/y,2.34,-0.080,2.348,10
Unemployment (% labour force),%,6.16,-0.336,-0.686,10
FDI inflow (% GDP),% GDP,4.30,0.336,-0.204,10
Political stability (z-score),z-score,0.54,-0.252,-0.457,10
Rule of law (z-score),z-score,1.07,-0.010,-0.044,10
Income inequality (Gini),index,33.90,-2.400,0.400,9
GDP per-capita growth (% y/y),% y/y,0.83,-0.275,9.127,10
Interest payments (% revenue),% revenue,5.41,-0.103,-2.337,10
"Political corruption index (0–1, higher = more corrupt)",index (0–1),0.17,0.000,0.045,10


## Step 2 — news

Four Google News queries per country (broad, government, economic, security),
de-duplicated by URL and scored by the keyword heuristic in
`article_ranking.score_relevance`. Anything under 0.3 is dropped as noise —
unless that leaves fewer than 3 articles, in which case the bar is relaxed
rather than returning an empty pane.

In [ ]:
items = article_enrichment.fetch_relevant_news(NAME, max_articles=MAX_ARTICLES)

print(f"{len(items)} articles after de-dupe, scoring, and the relevance cut")
pd.DataFrame([
    {
        "relevance": it.get("relevance_score"),
        "published": it.get("published"),
        "source": it.get("source"),
        "title": it.get("title"),
    }
    for it in items
])

2026-07-26 18:05:49,164 INFO    backend.utils.news_fetching.source_filter: Loaded 1 blocked news source(s).


2026-07-26 18:05:52,651 INFO    httpx: HTTP Request: GET https://wwd.com/pop-culture/celebrity-news/meghan-markle-prince-harry-portugal-summer-1239081815/ "HTTP/1.1 200 OK"


2026-07-26 18:05:52,852 INFO    httpx: HTTP Request: GET https://www.yahoo.com/news/science/articles/possible-great-white-shark-glides-163349088.html "HTTP/1.1 200 OK"


2026-07-26 18:05:53,060 INFO    httpx: HTTP Request: GET https://www.goodnewsnetwork.org/last-circus-elephant-in-portugal-welcomed-to-pangea-sanctuary/ "HTTP/1.1 200 OK"


2026-07-26 18:05:53,347 INFO    httpx: HTTP Request: GET https://www.nottinghamforest.co.uk/news/2026/july/26/forest-continue-pre-season-preparations-in-portugal "HTTP/1.1 200 OK"


...

2026-07-26 18:06:29,132 INFO    httpx: HTTP Request: GET https://www.japantimes.co.jp/sports/2026/07/07/soccer/world-cup/spain-portugal-world-cup/ "HTTP/1.1 200 OK"


2026-07-26 18:06:29,533 INFO    httpx: HTTP Request: GET https://www.reuters.com/sports/soccer/leao-replaces-felix-portugal-against-unchanged-croatia-2026-07-02/ "HTTP/1.1 401 HTTP Forbidden"


2026-07-26 18:06:30,609 INFO    httpx: HTTP Request: GET https://securitybrief.com.au/story/lampion-malware-campaign-targets-users-in-portugal "HTTP/1.1 200 OK"


20 articles after de-dupe, scoring, and the relevance cut


,relevance,published,source,title
0,1.00,2026-07-20T07:00:00Z,Governo.it,President Meloni’s press statement with Prime Minister Montenegro of Portuga...
1,1.00,2026-07-16T22:47:00Z,XXV Governo Constitucional,State of the Nation: Government guarantees stability to continue transformin...
2,1.00,2026-07-16T17:03:00Z,XXV Governo Constitucional,Luís Montenegro: The Government is delivering on its commitment to always do...
3,1.00,2026-07-16T07:00:00Z,Macau Business,Portugal: Government will do all it can to protect Sines refinery – minister...
4,1.00,2026-07-15T12:51:00Z,XXV Governo Constitucional,Prime Minister: Turning knowledge into development - XXV Governo Constitucional
5,1.00,2026-07-08T15:46:01Z,The Mighty 790 KFGO,IMF says hopes to engage on central banks’ changes to forward guidance - The...
6,1.00,2026-07-01T07:00:00Z,PBS,"Federal Reserve Chair Warsh emphasizes political independence, signals focus..."
7,1.00,2026-07-01T07:00:00Z,Anadolu Ajansı,"Fed chair says US central bank charting ‘new course,’ repeats no forward gui..."
8,1.00,2026-07-01T07:00:00Z,Investopedia,"Fed Chair Warsh Says Inflation Is Too High, But Risks Have Diminished Lately..."
9,1.00,2026-06-29T07:00:00Z,European Central Bank,Back to basics in an uncertain environment - European Central Bank


### Resolve and enrich

Google News links are redirect wrappers; these get unwrapped to real publisher
URLs, denylisted sources are dropped, and one GET per article recovers a
summary, body text, and thumbnail. Each survivor then gets the stable id
(`a1`, `a2`, …) the model refers back to.

In [ ]:
before_count = len(items)
items = article_enrichment.resolve_and_enrich(items, ISO2)

# Stable ids for the model to cite back (pipeline.py:128-129).
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"

print(f"{len(items)}/{before_count} survived the source denylist")
pd.DataFrame([
    {
        "id": it["id"],
        "source": it.get("source"),
        "words": len((it.get("content") or it.get("text") or "").split()),
        "image": bool(it.get("image")),
        "title": it.get("title"),
    }
    for it in items
]).set_index("id")

20/20 survived the source denylist


,source,words,image,title
id,,,,
a1,Governo.it,1004,True,President Meloni’s press statement with Prime Minister Montenegro of Portuga...
a2,XXV Governo Constitucional,462,True,State of the Nation: Government guarantees stability to continue transformin...
a3,XXV Governo Constitucional,530,True,Luís Montenegro: The Government is delivering on its commitment to always do...
a4,Macau Business,498,True,Portugal: Government will do all it can to protect Sines refinery – minister...
a5,XXV Governo Constitucional,608,True,Prime Minister: Turning knowledge into development - XXV Governo Constitucional
a6,The Mighty 790 KFGO,481,True,IMF says hopes to engage on central banks’ changes to forward guidance - The...
a7,PBS,773,True,"Federal Reserve Chair Warsh emphasizes political independence, signals focus..."
a8,Anadolu Ajansı,470,True,"Fed chair says US central bank charting ‘new course,’ repeats no forward gui..."
a9,Investopedia,1087,False,"Fed Chair Warsh Says Inflation Is Too High, But Risks Have Diminished Lately..."


## Step 2b — stage-1 digests

The scorer never sees raw article bodies. A cheap model (`gpt-4o-mini`) reads
each article's **full text** and returns a strict-JSON factual extraction plus a
0-100 `stage1_severity`. Every digest reaches the scorer; only the three
highest-severity articles are pasted in full.

Digests are cached in Postgres by `(country, as_of, url)` plus a hash of the
digested text, so re-running this cell on the same day costs nothing. `as_of` is
read from the payload the same way `upsert_snapshot` reads it, so the cache key
matches the snapshot's.

Nothing here raises: a per-article failure leaves `digest=None` (that article
degrades to its title and summary in the prompt), and a cache read/write problem
just means "no cache".

In [ ]:
AS_OF = data_push.payload_as_of(payload)   # the date upsert_snapshot would key on
items = digest_engine.digest_articles(items, country_display=NAME, iso2=ISO2, as_of=AS_OF)
fulltext_ids = digest_engine.select_fulltext_ids(items)

print(f"as_of={AS_OF}   full text goes to the scorer for: {fulltext_ids}")
pd.DataFrame([
    {
        "id": it["id"],
        "severity": it.get("stage1_severity"),
        "full_text": it["id"] in fulltext_ids,
        "about_country": (it.get("digest") or {}).get("directly_about_country"),
        "what_happened": (it.get("digest") or {}).get("what_happened"),
    }
    for it in items
]).sort_values("severity", ascending=False).set_index("id")

2026-07-26 18:06:45,464 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-07-26 18:06:45,825 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-07-26 18:06:46,395 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-07-26 18:06:46,531 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-07-26 18:06:46,946 INFO    backend.utils.ai.digest_engine: [PT] digests: ok=4 cached=16 failed=0


as_of=2026-07-26   full text goes to the scorer for: ['a3', 'a17', 'a1']


,severity,full_text,about_country,what_happened
id,,,,
a3,40.0,True,True,Prime Minister Luís Montenegro highlighted the reforms implemented by the Go...
a17,40.0,True,True,"A Portuguese startup, GTechPlasma, claims its new graphene-based material co..."
a2,25.0,False,True,The Government of Portugal emphasized the importance of political stability ...
a1,25.0,True,True,President Meloni welcomed Prime Minister Montenegro of Portugal to Rome for ...
a4,25.0,False,True,The Portuguese Government is monitoring the merger of energy company Galp an...
a5,25.0,False,True,Prime Minister Luís Montenegro opened the 1st Science and Innovation Meeting...
a7,25.0,False,False,Federal Reserve Chair Kevin Warsh emphasized the central bank's independence...
a6,25.0,False,False,The International Monetary Fund plans to engage with central banks on change...
a12,25.0,False,True,Prime Minister Luís Montenegro hosted the President of Mozambique Daniel Cha...


### What the mini model actually did

Each stage-1 call is deliberately narrow: a few thousand characters of one
article go in, a few hundred characters of strict JSON come out. The mini model
is an **extraction engine**, not an analyst — it is told to use only the text in
front of it and to write `"not stated"` rather than fill a gap from outside
knowledge. It never sees the macro payload, the other articles, or the scoring
rubric, and it never produces a risk score.

The one judgement it makes is `stage1_severity`, and that number is used for
exactly one thing: choosing which three articles the scorer reads in full.

In [ ]:
digested = [it for it in items if it.get("digest")]
chars_in = sum(len(digest_engine.article_input_text(it)) for it in digested)
chars_out = sum(len(json.dumps(it["digest"], ensure_ascii=False)) for it in digested)

print(f"MINI MODEL  ({ai_client.DIGEST_MODEL_NAME})")
print(f"  {len(digested)} calls, one per article")
print(f"  read    {chars_in:>8,} chars of article text")
print(f"  wrote   {chars_out:>8,} chars of JSON   "
      f"({chars_out / max(chars_in, 1):.0%} of what it read)")

# The highest-severity article, end to end.
focus = max(digested, key=lambda it: it.get("stage1_severity") or 0.0)
sent = digest_engine.article_input_text(focus)
returned = json.dumps(focus["digest"], indent=2, ensure_ascii=False)

print(f"\n{'=' * 78}\nONE CALL IN FULL - article {focus['id']}: {focus.get('title', '')[:60]}\n{'=' * 78}")
print(f"\n--- IN: the article text it read ({len(sent):,} chars, first 800 shown) ---\n")
print(sent[:800] + ("..." if len(sent) > 800 else ""))
print(f"\n--- OUT: the entire digest it returned ({len(returned):,} chars) ---\n")
print(returned)

MINI MODEL  (gpt-4o-mini-2024-07-18)
  20 calls, one per article
  read      89,069 chars of article text
  wrote     11,039 chars of JSON   (12% of what it read)

ONE CALL IN FULL - article a3: Luís Montenegro: The Government is delivering on its commitm

--- IN: the article text it read (3,566 chars, first 800 shown) ---

Luís Montenegro: The Government is delivering on its commitment to always do more
During the State of the Nation debate, the Prime Minister said that Portugal is changing and highlighted the measures implemented by the Government
Prime Minister Luís Montenegro said that Portugal is changing as a result of the reforms implemented over the past two years. Opening the State of the Nation debate in Parliament, he said that the Government is delivering on the commitments it made to the Portuguese people.
Higher incomes, housing and opportunities for young people
Luís Montenegro highlighted the measures introduced to increase household incomes, including personal income t

## Step 3 — LLM scoring 💸

One structured-output call rating investor risk over the next 12 months, given
the macro payload, **every** article's digest, and the full text of the three
highest-severity ones.

It never raises. Without `OPENAI_API_KEY`, or on a network or parse failure, it
returns `score=None` and no article scores — the rest of the notebook still
runs, the tables are just empty. If Step 2b produced no digests at all, it logs
an ERROR and falls back to the old title-and-summary prompt rather than skipping
the country. A country caught by the sanctions gate in `legal_restrictions.yaml`
is forced to `1.0` after the model runs.

### What the scoring model actually receives

`country_llm_score` assembles three blocks into `AI_PROMPT`. This cell builds
them with the same internal helpers the function calls, so what prints here is
what gets sent:

- **EVIDENCE_JSON** — the macro payload from Step 1.
- **ARTICLE_DIGESTS_JSON** — every article, as `{id, source, published_at,
  title, digest, stage1_severity}`. No article bodies. An article whose digest
  failed appears in the old `summary` shape instead, which is how you spot a
  degraded one.
- **FULL_TEXT** — only the three articles Step 2b ranked highest, verbatim, each
  capped at `_MAX_FULLTEXT_CHARS` (12,000).

So the scorer sees *breadth* from the digests and *depth* on the few articles
that earned it. Unlike the mini model it does hold the rubric — bands, hard
rules, guardrails — and it is the only model that outputs a risk score.

In [ ]:
evidence_json = json.dumps(payload, ensure_ascii=False)
digests_json = langchain_llm._digests_to_json(items)          # the prompt's builders,
fulltext = langchain_llm._fulltext_block(items, fulltext_ids)  # not a re-implementation

print(f"SCORING MODEL  ({ai_client.MODEL_NAME})  -  1 call")
print(f"  EVIDENCE_JSON        {len(evidence_json):>8,} chars   macro payload")
print(f"  ARTICLE_DIGESTS_JSON {len(digests_json):>8,} chars   all {len(items)} articles, digests only")
print(f"  FULL_TEXT            {len(fulltext):>8,} chars   {len(fulltext_ids)} articles verbatim: {', '.join(fulltext_ids)}")

# For contrast: what every article's full text would have cost.
all_text = sum(len(digest_engine.article_input_text(it)) for it in items)
print(f"\n  (all {len(items)} bodies in full would be {all_text:,} chars - "
      f"the digests carry them in {len(digests_json):,})")

print(f"\n{'=' * 78}\nARTICLE_DIGESTS_JSON - one entry, {len(items)} of these\n{'=' * 78}\n")
print(json.dumps(json.loads(digests_json)[0], indent=2, ensure_ascii=False))
print(f"\n{'=' * 78}\nFULL_TEXT - first 700 chars\n{'=' * 78}\n")
print(fulltext[:700] + ("..." if len(fulltext) > 700 else ""))

print(f"\n{'=' * 78}\nDIVISION OF LABOR\n{'=' * 78}")
pd.DataFrame([
    {
        "stage": "1 · digest",
        "model": ai_client.DIGEST_MODEL_NAME,
        "calls": len(digested),
        "reads": "one article's full text, nothing else",
        "returns": "facts + stage1_severity 0-100",
        "decides": "which 3 articles stage 2 reads in full",
    },
    {
        "stage": "2 · score",
        "model": ai_client.MODEL_NAME,
        "calls": 1,
        "reads": "macro payload + all digests + 3 full texts",
        "returns": "score, subscores, per-article impact + topic_group",
        "decides": "the risk score, and the dashboard Top-3",
    },
]).set_index("stage")

SCORING MODEL  (gpt-4o-2024-08-06)  -  1 call
  EVIDENCE_JSON           2,759 chars   macro payload
  ARTICLE_DIGESTS_JSON   15,459 chars   all 20 articles, digests only
  FULL_TEXT              15,551 chars   3 articles verbatim: a3, a17, a1

  (all 20 bodies in full would be 89,069 chars - the digests carry them in 15,459)

ARTICLE_DIGESTS_JSON - one entry, 20 of these

{
  "id": "a1",
  "source": "Governo.it",
  "published_at": "2026-07-20",
  "title": "President Meloni’s press statement with Prime Minister Montenegro of Portugal - Governo.it",
  "digest": {
    "actors": "President Meloni and Prime Minister Montenegro discussed opportunities for cooperation between Italy and Portugal.",
    "numbers": "2025 trade relations reached almost EUR 10 billion; the visit occurred on 20 July 2026; the Mattei Plan for Africa involves 18 African nations.",
    "transmission": "bilateral cooperation agreement signed by Foreign Ministers Tajani and Rangel; discussions on EU agenda and migration

,model,calls,reads,returns,decides
stage,,,,,
1 · digest,gpt-4o-mini-2024-07-18,20,"one article's full text, nothing else",facts + stage1_severity 0-100,which 3 articles stage 2 reads in full
2 · score,gpt-4o-2024-08-06,1,macro payload + all digests + 3 full texts,"score, subscores, per-article impact + topic_group","the risk score, and the dashboard Top-3"


In [ ]:
llm_output = langchain_llm.country_llm_score(
    country_display=NAME,
    payload=payload,
    articles=items,
    fulltext_ids=fulltext_ids,
)

print(f"score: {llm_output.get('score')}\n")
print(llm_output.get("bullet_summary"))

2026-07-26 18:06:52,264 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


score: 0.35

Portugal's investor risk is low-moderate, driven by stable political conditions and ongoing government reforms. The country is enhancing its defense capabilities and international cooperation, notably with Italy. Economic indicators show moderate volatility, with inflation and unemployment rates stabilizing. Political stability is supported by government reforms in housing, education, and health. However, political corruption remains a concern, slightly impacting governance. Overall, Portugal's economic and political environment is conducive to investment, with no significant conflict or regulatory uncertainty expected in the near term.


In [ ]:
display(pd.Series(llm_output.get("subscores") or {}, dtype="float64").to_frame("subscore"))

pd.DataFrame(llm_output.get("news_article_scores") or [])

,subscore
conflict_war,0.10
political_stability,0.35
governance_corruption,0.30
macroeconomic_volatility,0.25
regulatory_uncertainty,0.20


,id,impact,topic_group
0,a1,0.25,portugal_italy_cooperation
1,a2,0.25,portugal_political_stability
2,a3,0.40,portugal_government_reforms
3,a4,0.25,portugal_energy_sector
4,a5,0.25,portugal_science_innovation
5,a6,0.25,global_monetary_policy
6,a7,0.25,global_inflation_focus
7,a8,0.00,global_monetary_policy
8,a9,0.25,global_inflation_focus
9,a10,0.00,ecb_monetary_policy


## Step 4 — Top-3 selection

The model groups articles covering the same underlying event into a shared
`topic_group`. With 3 or more distinct topics, the best article of each of the
top 3 topics wins — so the dashboard shows three *stories* rather than three
write-ups of one. With fewer topics, the remainder is backfilled by impact.

The table shows every candidate, not just the winners, so you can see what lost.

In [ ]:
imp_map, topic_map = article_ranking.impact_topic_maps(llm_output)
items_by_id = {it.get("id"): it for it in items if isinstance(it, dict) and it.get("id")}
top_ids = article_ranking.select_top_ids(items_by_id, imp_map, topic_map, ISO2)

print(f"top 3: {top_ids}  (from {len(items_by_id)} candidates, "
      f"{len(set(topic_map.values()))} distinct topics)")
pd.DataFrame([
    {
        "id": aid,
        "selected": aid in top_ids,
        "impact": imp_map.get(aid),
        "topic_group": topic_map.get(aid),
        "published": it.get("published"),
        "title": it.get("title"),
    }
    for aid, it in items_by_id.items()
]).sort_values("impact", ascending=False).set_index("id")

2026-07-26 18:06:52,290 INFO    backend.utils.news_fetching.article_ranking: [PT] AI identified 15 topics (used 1/article).


top 3: ['a3', 'a17', 'a16']  (from 20 candidates, 15 distinct topics)


,selected,impact,topic_group,published,title
id,,,,,
a3,True,0.40,portugal_government_reforms,2026-07-16T17:03:00Z,Luís Montenegro: The Government is delivering on its commitment to always do...
a17,True,0.40,portugal_defense_innovation,2026-07-24T15:35:00Z,Portugal is developing a graphene-based stealth material that could hide pla...
a2,False,0.25,portugal_political_stability,2026-07-16T22:47:00Z,State of the Nation: Government guarantees stability to continue transformin...
a1,False,0.25,portugal_italy_cooperation,2026-07-20T07:00:00Z,President Meloni’s press statement with Prime Minister Montenegro of Portuga...
a4,False,0.25,portugal_energy_sector,2026-07-16T07:00:00Z,Portugal: Government will do all it can to protect Sines refinery – minister...
a5,False,0.25,portugal_science_innovation,2026-07-15T12:51:00Z,Prime Minister: Turning knowledge into development - XXV Governo Constitucional
a7,False,0.25,global_inflation_focus,2026-07-01T07:00:00Z,"Federal Reserve Chair Warsh emphasizes political independence, signals focus..."
a6,False,0.25,global_monetary_policy,2026-07-08T15:46:01Z,IMF says hopes to engage on central banks’ changes to forward guidance - The...
a12,False,0.25,portugal_mozambique_relations,2026-07-17T16:04:00Z,Prime Minister reaffirms commitment to cooperation between Portugal and Moza...


## Step 5 — images for the Top-3

Chosen articles still missing a thumbnail get one more try through Crawlbase,
which renders JavaScript. It costs a credit per call, hence the Top-3-only
scope, and no-ops entirely without `CRAWLBASE_TOKEN`.

In [ ]:
before_images = {aid: items_by_id[aid].get("image") for aid in top_ids}
article_enrichment.enrich_top_images(top_ids, items_by_id)

pd.DataFrame([
    {"id": aid, "before": before_images[aid], "after": items_by_id[aid].get("image")}
    for aid in top_ids
]).set_index("id")

,before,after
id,,
a3,https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...,https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...
a17,https://www.vozpopuli.com/indux/en/wp-content/uploads/2026/07/portugal-graph...,https://www.vozpopuli.com/indux/en/wp-content/uploads/2026/07/portugal-graph...
a16,https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...,https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...


## Step 6 — the Top-3 rows

These are the rows that would be written to `risk_snapshot_article` and rendered
on the country page, followed by a rough preview of how they look there.

In [ ]:
top_articles = article_ranking.build_top_articles(top_ids, items_by_id, imp_map)
pd.DataFrame(top_articles).set_index("rank")

,id,url,title,source,published_at,impact,summary,image
rank,,,,,,,,
1,a3,https://portugal.gov.pt/en/gc25/communication/news/luis-montenegro-the-gover...,Luís Montenegro: The Government is delivering on its commitment to always do...,XXV Governo Constitucional,2026-07-16T17:03:00Z,0.40,Luís Montenegro: The Government is delivering on its commitment to always do...,https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...
2,a17,https://www.vozpopuli.com/indux/en/portugal-is-developing-a-graphene-based-s...,Portugal is developing a graphene-based stealth material that could hide pla...,Vozpopuli,2026-07-24T15:35:00Z,0.40,A Portuguese startup says its new graphene-based material could sharply redu...,https://www.vozpopuli.com/indux/en/wp-content/uploads/2026/07/portugal-graph...
3,a16,https://portugal.gov.pt/en/gc25/communication/news/portugal-strengthens-coop...,Portugal strengthens cooperation with Italy - XXV Governo Constitucional,XXV Governo Constitucional,2026-07-20T16:12:00Z,0.25,"Portugal strengthens cooperation with Italy On an official visit to Italy, P...",https://edge.sitecorecloud.io/centrodeges65c9-cegere0de-prod6279-8af9/media/...


In [ ]:
from IPython.display import HTML

HTML("".join(
    '<div style="display:flex;gap:12px;margin:12px 0;align-items:flex-start">'
    + (f'<img src="{a["image"]}" style="width:160px;border-radius:6px">' if a["image"] else "")
    + f'<div><b>#{a["rank"]} &middot; impact {a["impact"]}</b><br>'
      f'<a href="{a["url"]}" target="_blank">{a["title"]}</a><br>'
      f'<small>{a["source"]} &middot; {a["published_at"]}</small><br>'
      f'<small>{(a["summary"] or "")[:240]}</small></div></div>'
    for a in top_articles
))

## Step 7 — the snapshot payload (not written)

The pipeline would hand this dict to `data_push.upsert_snapshot`, which writes
`country`, `indicator`, `yearly_value`, `risk_snapshot`, and
`risk_snapshot_article`.

**This notebook stops here on purpose** — no snapshot write, so a run can never
overwrite today's real snapshot for this country. (Step 2b's `article_digest`
rows are the one exception, and nothing reads those but Step 2b itself.) Use
`backend/tests/live_country_check.py` when you want the write plus verification
and cleanup.

In [ ]:
snapshot = {**payload, "llm_output": llm_output, "top_articles": top_articles}

# What upsert_snapshot validates before it would open a transaction.
print(f"country={snapshot['country']}  as_of<-{snapshot['_meta']['generated_at']}  "
      f"indicators={len(snapshot['indicators'])}  articles={len(snapshot['top_articles'])}")
print(json.dumps(snapshot, indent=2, default=str, ensure_ascii=False)[:3000])

country=PT  as_of<-2026-07-26T22:05Z  indicators=9  articles=3
{
  "country": "PT",
  "latest_year": 2025,
  "indicators": {
    "Inflation (% y/y)": {
      "latest": 2.34,
      "Δ1y": -0.08,
      "Δ5y": 2.348,
      "series": {
        "2016": 0.61,
        "2017": 1.37,
        "2018": 0.99,
        "2019": 0.34,
        "2020": -0.01,
        "2021": 1.27,
        "2022": 7.83,
        "2023": 4.31,
        "2024": 2.42,
        "2025": 2.34
      }
    },
    "Unemployment (% labour force)": {
      "latest": 6.16,
      "Δ1y": -0.336,
      "Δ5y": -0.686,
      "series": {
        "2016": 11.06,
        "2017": 8.88,
        "2018": 6.97,
        "2019": 6.42,
        "2020": 6.85,
        "2021": 6.71,
        "2022": 6.14,
        "2023": 6.5,
        "2024": 6.5,
        "2025": 6.16
      }
    },
    "FDI inflow (% GDP)": {
      "latest": 4.3,
      "Δ1y": 0.336,
      "Δ5y": -0.204,
      "series": {
        "2015": 0.64,
        "2016": 3.56,
        "2017": 5.02,
     

## Summary

Everything the run produced, on one screen — the "did this make sense?" cell.

In [ ]:
print(f"{NAME} ({ISO2})  -  risk score {llm_output.get('score')}\n")
print(pd.Series(llm_output.get("subscores") or {}, dtype="float64").to_string(), "\n")
print(llm_output.get("bullet_summary"), "\n")
for a in top_articles:
    print(f"  #{a['rank']}  impact {a['impact']}  {a['title']}")
    print(f"      {a['source']} - {a['published_at']}")
    print(f"      {a['url']}\n")

Portugal (PT)  -  risk score 0.35

conflict_war                0.10
political_stability         0.35
governance_corruption       0.30
macroeconomic_volatility    0.25
regulatory_uncertainty      0.20 

Portugal's investor risk is low-moderate, driven by stable political conditions and ongoing government reforms. The country is enhancing its defense capabilities and international cooperation, notably with Italy. Economic indicators show moderate volatility, with inflation and unemployment rates stabilizing. Political stability is supported by government reforms in housing, education, and health. However, political corruption remains a concern, slightly impacting governance. Overall, Portugal's economic and political environment is conducive to investment, with no significant conflict or regulatory uncertainty expected in the near term. 

  #1  impact 0.4  Luís Montenegro: The Government is delivering on its commitment to always do more - XXV Governo Constitucional
      XXV Governo Cons